# M3 — Tiền Xử Lý, Gán Nhãn & Trích Xuất Đặc Trưng (Colab + Drive)

Chạy **sau** khi M1 (crawl arXiv + Semantic Scholar) và M2 (parsing TeX/BibTeX) đã hoàn thành.
Quy trình: GitHub branch → Drive → đọc dữ liệu M1/M2 → gán nhãn thủ công + tự động → trích xuất 14 đặc trưng → chia split → xuất Parquet.

Các bước:
1. **SET UP** — Clone code, cài thư viện, chạy tests.
2. **Mount Google Drive** — Kết nối Drive để đọc/ghi dữ liệu.
3. **Trỏ Drive** — Cấu hình đường dẫn, kiểm tra dữ liệu đầu vào, override `config`.
4. **Nạp dữ liệu** — Đọc `bibtex.json`, `references.json` từ M1/M2; `elements.jsonl` đọc trực tiếp từ ZIP M2 (cho `cite_count`).
5. **Gán nhãn thủ công** — Xuất CSV mẫu cho 5 bài báo, nhóm kiểm tra và import lại.
6. **Chạy pipeline** — Gán nhãn tự động, kiểm tra quota, chia split, xuất Parquet.
7. **Trực quan hóa** — Vẽ biểu đồ phân bố 14 đặc trưng cho Poster A2.
8. **Lưu về Drive** — Copy Parquet + biểu đồ vào Drive.

**Cần push code Python mới nhất lên nhánh của nhóm trước khi chạy notebook.**

## SET UP — Clone code từ branch GitHub và cài thư viện

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/tuandatedu/arXir_project.git"  # TODO: thay URL repo nhóm
BRANCH = "M3-matching"                                         # TODO: tên branch thật
PROJECT = Path("/content/project")
assert "OWNER" not in REPO_URL, "Bạn chưa thay URL GitHub"

if not (PROJECT / ".git").exists():
    subprocess.run(["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, str(PROJECT)], check=True)
else:
    subprocess.run(["git", "-C", str(PROJECT), "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(PROJECT), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(PROJECT), "pull", "--ff-only", "origin", BRANCH], check=True)

os.chdir(PROJECT)
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "src/requirements.txt"], check=True)
subprocess.run([sys.executable, "-m", "pytest", "-q", "tests/test_m3_matching.py"], check=True)
print("Repo:", PROJECT, "| Branch:", BRANCH)

## SET UP — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 1 — Trỏ đúng thư mục Drive và cấu hình đường dẫn

Chỉnh `RANGE_ID` cho khớp với range bạn phụ trách. Notebook sẽ đọc dữ liệu M1/M2 đã có trên Drive và ghi kết quả M3 vào thư mục riêng.

Cấu trúc Google Drive kỳ vọng:

```text
MyDrive/
├── references/
│   └── range_1/           # Kết quả S2 từ M1
│       ├── reference/     # *.json từng bài
│       └── tables/
│           ├── papers.json
│           └── references.json
├── M2/
│   └── range_1/           # Kết quả M2
│       └── M2_range_1_tables.zip   # gồm bibtex.jsonl + elements.jsonl
└── M3_matching/
    └── range_1/           # Output notebook này (MY_M3_DIR)
        ├── tables/
        │   ├── bibtex.json
        │   └── references.json
        ├── labels/
        │   ├── manual/
        │   │   ├── *.csv        # CSV mẫu để gán nhãn
        │   │   └── *.json       # nhãn manual sau khi import
        │   └── automatic/
        │       └── automatic.zip     # gộp nhãn auto
        │           └── *.json
        │       └── *.json
        └── prepared_dataset/
            ├── dataset_pairs.parquet
            ├── train_pairs.parquet
            ├── valid_pairs.parquet
            ├── test_pairs.parquet
            └── poster_feature_distribution.png
```

In [ ]:
RANGE_ID = 1  # TODO: đổi thành range của bạn

In [ ]:
import os
import zipfile
from datetime import datetime, timezone
from pathlib import Path

# 1. Thư mục dữ liệu M1 (S2 references) — chỉ đọc
M1_REFS_DIR = Path(
    f"/content/drive/MyDrive/references/range_{RANGE_ID}"
)
M1_REF_DIR = M1_REFS_DIR / "reference"          # chứa các file 2609-xxxxx.json
m1_refs_json = M1_REFS_DIR / "tables" / "references.json"

# 2. Dữ liệu M2 (bibtex parsed) — chỉ đọc
M2_DIR = Path(
    f"/content/drive/MyDrive/M2/range_{RANGE_ID}"
)
M2_ZIP = M2_DIR / f"M2_range_{RANGE_ID}_tables.zip"

# 3. Thư mục output M3 — đọc/ghi
MY_M3_DIR = Path(
    f"/content/drive/MyDrive/M3_matching/range_{RANGE_ID}"
)
TABLES_DIR = MY_M3_DIR / "tables"
LABELS_DIR = MY_M3_DIR / "labels"
PREPARED_DIR = MY_M3_DIR / "prepared_dataset"

for path in (MY_M3_DIR, TABLES_DIR, LABELS_DIR, LABELS_DIR / "manual", LABELS_DIR / "automatic", PREPARED_DIR):
    path.mkdir(parents=True, exist_ok=True)

# 4. Kiểm tra dữ liệu đầu vào từ M1/M2
m1_ref_files = sorted(M1_REF_DIR.glob("*.json"))
assert m1_ref_files, (
    f"Không có file *.json nào trong {M1_REF_DIR} "
    "— kiểm tra lại đường dẫn Drive và kết quả M1"
)

assert m1_refs_json.exists(), f"Chưa tìm thấy {m1_refs_json} — kiểm tra lại kết quả M1"
assert M2_DIR.exists(), f"Chưa tìm thấy {M2_DIR} — kiểm tra lại đường dẫn Drive và kết quả M2"
M2_BIB_CANDIDATES = [M2_DIR / "tables" / "bibtex.jsonl", M2_DIR / "tables" / "bibtex.json"]
m2_bib_source = next((p for p in M2_BIB_CANDIDATES if p.exists()), None)
assert m2_bib_source or M2_ZIP.exists(), (
    f"Không có bibtex.jsonl/bibtex.json trong {M2_DIR} và cũng không có {M2_ZIP}"
)
m2_bibtex_json = TABLES_DIR / "bibtex.json"
if not m2_bibtex_json.exists():
    if m2_bib_source:
        m2_bibtex_json.write_bytes(m2_bib_source.read_bytes())
    else:
        with zipfile.ZipFile(M2_ZIP) as z:
            names = [n for n in z.namelist() if n.endswith(("bibtex.jsonl", "bibtex.json"))]
            assert names, f"Không có bibtex.jsonl trong {M2_ZIP}: {z.namelist()[:10]}"
            m2_bibtex_json.write_bytes(z.read(names[0]))
assert m2_bibtex_json.exists(), f"Chưa tìm thấy {m2_bibtex_json}"

# 4b. elements.jsonl đọc trực tiếp trong ZIP M2 (không copy vào M3)
if M2_ZIP.exists():
    print(f"M2 elements    : {M2_ZIP} (đọc trực tiếp elements.jsonl trong ZIP)")
else:
    print("M2 elements    : KHÔNG CÓ ZIP -> cite_count sẽ = 0")

# 5. Run ID
RUN_ID = os.environ.get(
    "M3_RUN_ID",
    datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
)

# 6. Override config và metrics để pipeline ghi đúng vào Drive
from src.common import config
import src.common.metrics as metrics_module

config.WORKING_DIR = MY_M3_DIR
config.TABLES_DIR  = TABLES_DIR
metrics_module.TABLES_DIR = TABLES_DIR

print(f"Range          : {RANGE_ID}")
print(f"M1 refs từ     : {M1_REF_DIR} ({len(m1_ref_files)} file)")
print(f"M2 bibtex từ   : {m2_bib_source or M2_ZIP} -> {m2_bibtex_json}")
print(f"M3 output tại  : {MY_M3_DIR}")
print(f"TABLES_DIR     : {config.TABLES_DIR}")
print(f"LABELS_DIR     : {LABELS_DIR}")
print(f"RUN_ID         : {RUN_ID}")

## 2 — Nạp dữ liệu từ Drive (M1 References + M2 BibTeX)

Copy `bibtex.json` từ M2 và `references.json` từ M1 vào `TABLES_DIR`; `elements.jsonl` được đọc trực tiếp từ ZIP M2 khi chạy pipeline.
Dữ liệu gốc trên Drive **không bị sửa**.

In [ ]:
import json
from pathlib import Path

def _load_records(path):
    text = Path(path).read_text(encoding="utf-8-sig")
    if text.lstrip().startswith("["):
        return json.loads(text)
    return [json.loads(line) for line in text.splitlines() if line.strip()]

def _write_json(path, records):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        json.dump(records, f, ensure_ascii=False, indent=2)

# --- 1. Copy bibtex.json từ M2 ---
dst_bib = TABLES_DIR / "bibtex.json"
bib_records = _load_records(m2_bibtex_json)
_write_json(dst_bib, bib_records)
print(f"✓ bibtex.json  : {len(bib_records):,} bản ghi  ({dst_bib})")

# --- 2. Chuyển references.json (M1) → references.json ---
dst_refs = TABLES_DIR / "references.json"
ref_records = _load_records(m1_refs_json)
_write_json(dst_refs, ref_records)
print(f"✓ references.json: {len(ref_records):,} bản ghi  ({dst_refs})")

print(f"\nDữ liệu đầu vào đã sẵn sàng tại: {TABLES_DIR}")

## 3 — Hỗ trợ gán nhãn thủ công

Tìm các bài báo có $\ge 20$ BibTeX entries, xuất CSV mẫu kèm gợi ý tự động để nhóm xác nhận.
Sau khi nhóm điền xong file CSV, dùng `import_manual_csv` để nạp nhãn vào `LABELS_DIR`.

Yêu cầu tối thiểu: **5 bài báo**, mỗi bài $\ge 20$ entries, tổng $\ge 100$ entries.

In [ ]:
import pandas as pd
from src.m3_matching.manual_tool import (
    export_manual_template,
    find_candidate_manual_papers,
    import_manual_csv,
    summarize_team_labels,
)

# 1. Tìm bài đủ điều kiện
candidates = find_candidate_manual_papers(
    bibtex_path=TABLES_DIR / "bibtex.json",
    references_path=TABLES_DIR / "references.json",
)
print(f"Tìm thấy {len(candidates)} bài có ≥ 20 BibTeX entries và có references:")
for pid, b_cnt, r_cnt in candidates[:10]:
    print(f"  - {pid}: {b_cnt} BibTeX | {r_cnt} References")

assert len(candidates) >= 5, (
    f"Chỉ có {len(candidates)} bài đủ điều kiện (cần ≥ 5). "
    "Kiểm tra lại dữ liệu M1/M2."
)

# 2. Xuất CSV mẫu cho 5 bài đầu tiên
TEMPLATE_DIR = LABELS_DIR / "manual"
TEMPLATE_DIR.mkdir(parents=True, exist_ok=True)
selected_5 = [pid for pid, _, _ in candidates[:5]]

for pid in selected_5:
    out_csv = TEMPLATE_DIR / f"{pid}.csv"
    export_manual_template(
        pid, out_csv,
        bibtex_path=TABLES_DIR / "bibtex.json",
        references_path=TABLES_DIR / "references.json",
    )
    print(f"  -> Đã xuất template: {out_csv}")

print(f"\n✓ Đã xuất {len(selected_5)} file CSV mẫu tại: {TEMPLATE_DIR}")

# 3. Xem trước bảng mẫu đầu tiên
demo_pid = selected_5[0]
df_preview = pd.read_csv(TEMPLATE_DIR / f"{demo_pid}.csv")
display(df_preview[[
    "citation_key", "bib_title",
    "suggested_ref_title", "suggested_score", "groundtruth_arxiv_id"
]].head(10))

In [ ]:
# Chạy cell này SAU KHI nhóm đã điền xong file CSV và upload lên Drive.
# Mỗi thành viên import file của mình; cell có thể chạy nhiều lần.

for pid in selected_5:
    csv_path = TEMPLATE_DIR / f"{pid}.csv"
    if csv_path.exists():
        out = import_manual_csv(csv_path, pid, labels_dir=LABELS_DIR)
        print(f"  ✓ Đã import nhãn thủ công {pid} → {out}")
    else:
        print(f"  [skip] Chưa có file: {csv_path}")

summary = summarize_team_labels(labels_dir=LABELS_DIR)
print("\n=== TIẾN ĐỘ GÁN NHÃN ===")
print(f"  Bài manual   : {summary['n_manual_papers']} (yêu cầu ≥ 5)")
print(f"  Entries manual: {summary['n_manual_entries']} (yêu cầu ≥ 100)")
print(f"  Quota status : {summary['quota_status']}")

## 4 — Chạy trọn gói pipeline M3

Thực hiện theo thứ tự:
- Gán nhãn tự động cho $\ge 10\%$ bài ngoài nhóm manual (`labelling.py`)
- Kiểm tra quota (`validate_label_quota`)
- Chia tập `train` / `valid` / `test` theo bài báo (`split.py`)
- Sinh bảng cặp ứng viên kèm 14 đặc trưng và nhãn 0/1 (`dataset.py`)
- Xuất 4 file Parquet vào `TABLES_DIR`

In [ ]:
from src.m3_matching.prepare_dataset import run_preparation_pipeline

K_MEMBERS  = 1   # Mỗi range là dataset riêng của 1 người -> luôn để 1
RANDOM_SEED = 42

summary = run_preparation_pipeline(
    tables_dir=TABLES_DIR,
    labels_dir=LABELS_DIR,
    k=K_MEMBERS,
    seed=RANDOM_SEED,
    demo=False,
    elements_zip=M2_ZIP,
    zip_auto_labels=True,
)

print("\n=================== KẾT QUẢ XUẤT BẢN DỮ LIỆU ===================")
print(f"  Số đặc trưng  : {len(summary['feature_names'])} features")
print(f"  Train : {summary['splits']['train']['n_pairs']:,} cặp "
      f"({summary['splits']['train']['n_positives']} pos, "
      f"{summary['splits']['train']['n_negatives']} neg)")
print(f"  Valid : {summary['splits']['valid']['n_pairs']:,} cặp "
      f"({summary['splits']['valid']['n_positives']} pos, "
      f"{summary['splits']['valid']['n_negatives']} neg)")
print(f"  Test  : {summary['splits']['test']['n_pairs']:,} cặp "
      f"({summary['splits']['test']['n_positives']} pos, "
      f"{summary['splits']['test']['n_negatives']} neg)")
print("==================================================================")

print("\nCác file đã xuất:")
for f in summary["exported_files"]:
    p = Path(f)
    if p.exists():
        print(f"  ✓ {p.name}: {p.stat().st_size:,} bytes")
    else:
        print(f"  ✗ {p.name}: MISSING")

## 5 — Trực quan hóa 14 đặc trưng (cho Poster A2)

Vẽ biểu đồ phân bố 3 đặc trưng đại diện giữa nhãn 0 và nhãn 1 trên tập train.
File ảnh được lưu vào Drive cùng với Parquet.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# ==========================================
# 1. TẠO BIỂU ĐỒ FEATURE DISTRIBUTION
# ==========================================

train_df = pd.read_parquet(PREPARED_DIR / "train_pairs.parquet")
train_df["label"] = pd.to_numeric(train_df["label"], errors="coerce")

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Title similarity
sns.histplot(
    data=train_df,
    x="title_token_sort",
    hue="label",
    bins=30,
    kde=True,
    common_norm=False,
    stat="density",
    ax=axes[0]
)
axes[0].set_title("Title Token Sort Similarity")

# 2. Author similarity
sns.histplot(
    data=train_df,
    x="author_jaccard",
    hue="label",
    bins=30,
    kde=True,
    common_norm=False,
    stat="density",
    ax=axes[1]
)
axes[1].set_title("Author Jaccard Similarity")

# 3. Year difference
year_df = train_df.copy()
year_df["year_diff"] = pd.to_numeric(
    year_df["year_diff"], errors="coerce"
)
year_df = year_df[
    year_df["label"].isin([0, 1])
    & year_df["year_diff"].between(0, 19, inclusive="left")
].copy()
year_df["label"] = year_df["label"].astype(int)

sns.boxplot(
    data=year_df,
    x="label",
    y="year_diff",
    hue="label",
    ax=axes[2]
)
axes[2].set_title("Publication Year Difference")
axes[2].set_xticks([0, 1])
axes[2].set_xticklabels(["NO (0)", "YES (1)"])

plt.tight_layout()

# Lưu biểu đồ vào Drive
poster_path = PREPARED_DIR / "poster_feature_distribution.png"
plt.savefig(poster_path, dpi=300, bbox_inches="tight")
plt.show()
plt.close()

print(f"✓ Đã lưu biểu đồ: {poster_path}")


## 6 — Lưu kết quả về Google Drive

Copy toàn bộ file Parquet và ảnh biểu đồ từ `TABLES_DIR` vào Drive.
Vì `TABLES_DIR` đã **là thư mục Drive** (`MY_M3_DIR/tables/`), dữ liệu đã được lưu trực tiếp trong quá trình pipeline. Cell này chỉ kiểm tra và in xác nhận.

In [ ]:
import zipfile

# ==========================================
# 2. KIỂM TRA TOÀN BỘ FILE OUTPUT
# ==========================================

export_jsonl = [
    "bibtex.json",
    "references.json",
]

export_prepared = [
    "train_pairs.parquet",
    "valid_pairs.parquet",
    "test_pairs.parquet",
    "dataset_pairs.parquet",
    "poster_feature_distribution.png",
]

print(f"Kiểm tra kết quả trong: {MY_M3_DIR}")
print(f"Run ID: {RUN_ID}\n")

all_ok = True
for folder, names in (
    (TABLES_DIR, export_jsonl),
    (PREPARED_DIR, export_prepared)
):
    for fname in names:
        p = folder / fname
        if p.exists():
            print(f"  ✓ {fname}: {p.stat().st_size:,} bytes")
        else:
            print(f"  ✗ {fname}: MISSING")
            all_ok = False

# Kiểm tra automatic.zip
auto_zip = LABELS_DIR / "automatic" / "automatic.zip"
if auto_zip.exists():
    with zipfile.ZipFile(auto_zip) as zf:
        names_in_zip = zf.namelist()
    print(f"  ✓ labels/automatic/automatic.zip: {auto_zip.stat().st_size:,} bytes ({len(names_in_zip)} files)")
else:
    print(f"  [info] labels/automatic/automatic.zip: chưa có (chỉ được tạo nếu có auto labels)")

print()
if all_ok:
    print("HOÀN TẤT TOÀN BỘ QUY TRÌNH MILESTONE 3!")
else:
    print("[warn] Một số file bị thiếu — kiểm tra lại các bước trên.")


## 7 — Thống kê 14 đặc trưng & Phân bổ bài báo (Cho Poster A2 + Justification)

Tính toán thống kê min/mean/max/std của 14 đặc trưng trên tập train.
Phân bổ số bài báo: bao nhiêu từ nhãn thủ công, bao nhiêu từ tự động gán.
Giúp chứng minh rằng các đặc trưng được lựa chọn có ý nghĩa thống kê và được xác thực trên dữ liệu thực.

**14 Đặc trưng:**
1. `title_levenshtein` — Khoảng cách Levenshtein tiêu đề (normalized)
2. `title_token_sort` — Token Sort Ratio tiêu đề
3. `title_token_set` — Token Set Ratio tiêu đề
4. `title_jaccard` — Jaccard similarity tiêu đề
5. `title_exact` — Boolean: tiêu đề trùng khớp chính xác
6. `title_len_diff_ratio` — Tỉ lệ độ dài tiêu đề khác nhau
7. `author_jaccard` — Jaccard similarity danh sách tác giả
8. `author_first_match` — Boolean: tác giả đầu tiên trùng khớp
9. `author_any_match` — Boolean: bất kì tác giả nào trùng khớp
10. `year_diff` — Chênh lệch năm công bố (tuyệt đối)
11. `year_exact` — Boolean: năm công bố trùng khớp chính xác
12. `year_missing` — Boolean: năm bị thiếu trong một trong hai bài
13. `doi_match` — Boolean: DOI trùng khớp
14. `cite_count` — Số lần được trích dẫn (nếu có)

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# 1. Tải dữ liệu train
train_df = pd.read_parquet(PREPARED_DIR / "train_pairs.parquet")
print(f"Tập train: {len(train_df):,} cặp tham chiếu")

# 2. Danh sách 14 đặc trưng
FEATURE_NAMES = [
    "title_levenshtein",
    "title_token_sort",
    "title_token_set",
    "title_jaccard",
    "title_exact",
    "title_len_diff_ratio",
    "author_jaccard",
    "author_first_match",
    "author_any_match",
    "year_diff",
    "year_exact",
    "year_missing",
    "doi_match",
    "cite_count",
]

# 3. Kiểm tra tất cả đặc trưng có trong train_df
missing_features = [f for f in FEATURE_NAMES if f not in train_df.columns]
if missing_features:
    print(f"[WARN] Các đặc trưng bị thiếu: {missing_features}")
    FEATURE_NAMES = [f for f in FEATURE_NAMES if f in train_df.columns]

print(f"Số đặc trưng trong train_df: {len(FEATURE_NAMES)} / 14")

# 4. Tính toán thống kê cho từng đặc trưng
stats_list = []
for feat in FEATURE_NAMES:
    values = train_df[feat].dropna()
    stats_list.append({
        "Feature": feat,
        "Min": values.min(),
        "Mean": values.mean(),
        "Std": values.std(),
        "Max": values.max(),
        "Count": len(values),
    })

stats_df = pd.DataFrame(stats_list)
display(stats_df)


In [ ]:
import pandas as pd

# Đọc cả 3 tập
all_papers = set()
for filename in [
    "train_pairs.parquet",
    "valid_pairs.parquet",
    "test_pairs.parquet",
]:
    df = pd.read_parquet(PREPARED_DIR / filename)
    # Sử dụng đúng tên cột theo schema
    id_col = "arxiv_id" if "arxiv_id" in df.columns else "paper_id"
    if id_col not in df.columns:
        raise ValueError(f"Không tìm thấy cột arxiv_id/paper_id trong {filename}")
    all_papers.update(df[id_col].dropna().astype(str).unique())

# Lấy Manual publications
manual_dir = LABELS_DIR / "manual"
manual_papers = {
    f.stem for f in manual_dir.glob("*.json")
} if manual_dir.exists() else set()

# Phân loại
n_manual = len(manual_papers & all_papers)
n_automatic = len(all_papers - manual_papers)
n_total = len(all_papers)

papers_breakdown = pd.DataFrame([
    {
        "Category": "Manual",
        "Papers": n_manual,
        "Percentage": f"{100*n_manual/n_total:.1f}%" if n_total else "0%",
    },
    {
        "Category": "Automatic",
        "Papers": n_automatic,
        "Percentage": f"{100*n_automatic/n_total:.1f}%" if n_total else "0%",
    },
    {
        "Category": "Total",
        "Papers": n_total,
        "Percentage": "100%",
    },
])

display(papers_breakdown)


In [ ]:
# 8. Tóm tắt cho Poster A2

print("\n" + "="*70)
print("THỐNG KÊ POSTER A2 — M3 MATCHING")
print("="*70)

print(f"\n📊 DỮ LIỆU HUẤN LUYỆN:")
print(f"  - Tổng cặp tham chiếu: {len(train_df):,}")
print(f"  - Cặp trùng khớp (label=1): {(train_df['label'] == 1).sum():,}")
print(f"  - Cặp khác bài (label=0): {(train_df['label'] == 0).sum():,}")
print(f"  - Tỉ lệ trùng khớp: {100 * (train_df['label'] == 1).sum() / len(train_df):.1f}%")

print(f"\n🗂️ PHÂN BỐ BÀI BÁO:")
print(f"  - Bài báo từ nhãn thủ công: {n_manual} ({100*n_manual/n_total:.1f}%)")
print(f"  - Bài báo từ nhãn tự động: {n_automatic} ({100*n_automatic/n_total:.1f}%)")
print(f"  - Tổng: {n_total} bài báo")

print(f"\n✨ CÁC ĐẶC TRƯNG (14 features):")
for i, row in stats_df.iterrows():
    feat = row["Feature"]
    mean_val = row["Mean"]
    print(f"  {i+1:2d}. {feat:25s} — mean={mean_val:6.3f}, std={row['Std']:6.3f}")

print(f"\n💾 FILE ĐÃ LƯU TRONG {PREPARED_DIR}:")
print(f"  - train_pairs.parquet")
print(f"  - valid_pairs.parquet")
print(f"  - test_pairs.parquet")
print(f"  - dataset_pairs.parquet")
print(f"  - poster_feature_distribution.png")

print("\n" + "="*70)
